# 07. Fleet-wide Volt-VAr slope analysis 

1. Polarity-corrected net meter reading (`ami_meter_residential` × `circuit_polarity`).
2. Night-time baseline, **per circuit and per month**: `P_est = base_P − P_net`, `Q_est = base_Q − Q_net`.
3. **Signed net-export gate**: keep intervals with `P_est ≥ 20%` of rating (|P| would also admit household import).
4. OLS of `Q_est` (% of rating) on `V` over the assessable window, e.g. [240, 253) V.

**Sites**: `n_load_phases == 1` and `n_pv_phases == 1`, excluding `flex_export_detected`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Put the repo root on sys.path (same as the other notebooks).
_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import pandas as pd
import numpy as np

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_conformance_plot as ConformancePlot

## 1. Settings

In [ ]:
MONTHS = [
    f"{year}-{month:02d}"
    for year in (2024, 2025)
    for month in range(1, 13)
]

# Method settings (defaults = the values validated in notebook 05)
NIGHT_HOURS = (0, 4)             # AEST hours for the monthly night-time baseline
P_MIN_FRAC = Fleet.P_MIN_FRAC    # signed gate: P_est >= 20% of rating
#V_LO, V_HI = Fleet.V_LO, Fleet.V_HI 

V_LO = 242
V_HI = 253

FIT_SETTINGS = dict(night_hours=NIGHT_HOURS, p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI)

# Fit-quality thresholds applied AFTER the run
MIN_N_WINDOW = 20       # min. 5-min intervals in the assessable window per site-month 
MIN_V_SPREAD = 5.0       # min. voltage range (V) covered by those intervals

# Pass/fail tolerance for the slope of the fitted line (V/kW) relative to the expected slope (V/kW)
# Goes both positive and negative, e.g. 20% means the slope must be within 80% to 120% of the expected slope
SLOPE_TOLERANCE_PCT = 20

# DuckDB resources
DUCKDB_MEMORY, DUCKDB_THREADS = "8GB", 4

# Outputs live next to the data store 
OUT_DIR = Fleet.store_dir() / "vvar_fleet" / f"2024_2025_band_{V_LO}_{V_HI}"
print(f"Results folder: {OUT_DIR}")

## 2. Site eligibility

In [ ]:
site_meta = pd.read_parquet(Fleet.Config.store_path("ami_site_metadata_residential"))
eligibility = Fleet.build_eligibility(site_meta)
sites = eligibility.loc[eligibility.eligibility == "eligible"]

display(eligibility.eligibility.value_counts().rename("n_sites").to_frame())
print(f"{len(sites):,} eligible sites (1 load phase, 1 PV phase, no flex export, export limit OK).")

## 3. Run, month by month (resumable)

In [ ]:
# Blind analysis: net meter + monthly baseline + signed gate
log_blind = Fleet.run_months(
    MONTHS, OUT_DIR / "blind", sites, source="meter_baseline",
    memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **FIT_SETTINGS,
)
display(log_blind)

In [ ]:
# Ground truth: the separate PV circuit, same window and gate (no baseline needed)
log_truth = Fleet.run_months(
    MONTHS, OUT_DIR / "truth", sites, source="ground_truth",
    memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **FIT_SETTINGS,
)
display(log_truth)

## 4. Load results and apply fit-quality thresholds

In [ ]:
blind = Fleet.add_fit_status(Fleet.load_results(OUT_DIR / "blind"), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
truth = Fleet.add_fit_status(Fleet.load_results(OUT_DIR / "truth"), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)

# Reference data quality (notebook 06a): PV circuits reporting constant Q are not assessable
constant_q = (truth.fit_status == "ok") & (truth.slope == 0) & (truth.slope_stderr == 0)
truth.loc[constant_q, "fit_status"] = "constant_q"
print(f"{constant_q.sum():,} reference site-months set to 'constant_q' "
      f"({truth.loc[constant_q, 'site_id'].nunique():,} sites)")

# Sanity check: one row per site-month (a single-phase site should have one circuit)
for name, df in [("blind", blind), ("truth", truth)]:
    n_dup = df.duplicated(["site_id", "year", "month"]).sum()
    assert n_dup == 0, f"{name}: {n_dup} duplicated site-months -- a site has >1 circuit"

# Site-months by fit status, per month
display(pd.crosstab([blind.year, blind.month], blind.fit_status, margins=True))
display(pd.crosstab([truth.year, truth.month], truth.fit_status, margins=True))

## 5. Validation against ground truth

In [ ]:
compare = Fleet.compare_to_truth(blind, truth)
err = compare.slope_err.abs()
print(f"{len(compare):,} site-months with both fits ok")
print(f"|slope error| (%/V): median {err.median():.3f}, p75 {err.quantile(.75):.3f}, "
      f"p90 {err.quantile(.9):.3f}, share < 0.5 %/V: {(err < 0.5).mean():.1%}")

Fleet.plot_blind_vs_truth(compare)
plt.show()

In [ ]:
# Worst disagreements -- candidates for a site-level flag (see the Section-12 tail sites)
display(compare.sort_values("slope_err", key=abs, ascending=False).head(15).round(3))

## 6. Fleet results

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), dpi=130, sharey=True)
Fleet.plot_monthly_slopes(blind, "Blind (net meter + monthly baseline + signed gate)", ax=axes[0])
Fleet.plot_monthly_slopes(truth, "Ground truth (separate PV circuit)", ax=axes[1])
fig.tight_layout()
plt.show()

In [ ]:
# Level/offset of each fitted line vs the required line at V3, V4 and their midpoint
blind_ok = ConformancePlot.add_offset_pct_columns(blind.loc[blind.fit_status == "ok"])
truth_ok = ConformancePlot.add_offset_pct_columns(truth.loc[truth.fit_status == "ok"])

summary = pd.concat([
    df.groupby(["year", "month"]).agg(n_sites=("site_id", "nunique"), median_slope=("slope", "median"),
                                      median_r2=("r_squared", "median"), median_offset_mid=("offset_pct_mid", "median"))
      .assign(source=name)
    for name, df in [("blind", blind_ok), ("truth", truth_ok)]
]).reset_index().pivot(index=["year", "month"], columns="source")
display(summary.round(3))

In [ ]:
# Pass/fail on slope (runs once SLOPE_TOLERANCE_PCT is set in Section 1)
if SLOPE_TOLERANCE_PCT is None:
    print("SLOPE_TOLERANCE_PCT not set yet -- skipping pass/fail.")
else:
    blind_pf = ConformancePlot.add_slope_pass_fail(blind_ok, SLOPE_TOLERANCE_PCT)
    truth_pf = ConformancePlot.add_slope_pass_fail(truth_ok, SLOPE_TOLERANCE_PCT)

    # Monthly pass rate
    pass_rate = pd.DataFrame({
        "blind_%": blind_pf.groupby(["year", "month"]).passed.mean() * 100,
        "truth_%": truth_pf.groupby(["year", "month"]).passed.mean() * 100,
    })
    display(pass_rate.round(1))

    # Agreement: does the blind pass/fail match the ground-truth pass/fail for the same site-month?
    keys = ["site_id", "year", "month"]
    both = blind_pf[keys + ["passed"]].merge(truth_pf[keys + ["passed"]], on=keys, suffixes=("_blind", "_truth"))
    display(pd.crosstab(both.passed_blind, both.passed_truth, rownames=["blind"], colnames=["truth"]))
    print(f"Agreement: {(both.passed_blind == both.passed_truth).mean():.1%} of {len(both):,} site-months")

In [ ]:
# Pass/fail on slope (runs once SLOPE_TOLERANCE_PCT is set in Section 1)
POSITIVE_IS_FAIL = False   # positive class = non-conformant (fails the slope test); False -> positive = passes

def classification_metrics(pred: pd.Series, actual: pd.Series) -> dict:
    """Confusion counts + accuracy/precision/recall/specificity/F1 (NaN where undefined)."""
    tp = int((pred & actual).sum());   fp = int((pred & ~actual).sum())
    fn = int((~pred & actual).sum());  tn = int((~pred & ~actual).sum())
    div = lambda a, b: a / b if b else float("nan")
    precision, recall = div(tp, tp + fp), div(tp, tp + fn)
    return {"n": tp + fp + fn + tn, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": div(tp + tn, tp + fp + fn + tn), "precision": precision, "recall": recall,
            "specificity": div(tn, tn + fp), "f1": div(2 * precision * recall, precision + recall)}

if SLOPE_TOLERANCE_PCT is None:
    print("SLOPE_TOLERANCE_PCT not set yet -- skipping pass/fail.")
else:
    blind_pf = ConformancePlot.add_slope_pass_fail(blind_ok, SLOPE_TOLERANCE_PCT)
    truth_pf = ConformancePlot.add_slope_pass_fail(truth_ok, SLOPE_TOLERANCE_PCT)

    # Monthly pass rate
    pass_rate = pd.DataFrame({
        "blind_%": blind_pf.groupby(["year", "month"]).passed.mean() * 100,
        "truth_%": truth_pf.groupby(["year", "month"]).passed.mean() * 100,
    })
    display(pass_rate.round(1))

    # Same site-month under both sources (ground truth = the reference label)
    keys = ["site_id", "year", "month"]
    cols = keys + ["passed", "slope", "pct_error", "r_squared", "n_window"]
    both = blind_pf[cols].merge(truth_pf[cols], on=keys, suffixes=("_blind", "_truth"))
    display(pd.crosstab(both.passed_blind, both.passed_truth, rownames=["blind"], colnames=["truth"]))

    # Positive class: fail (non-conformant) by default
    both["pred_pos"] = ~both.passed_blind if POSITIVE_IS_FAIL else both.passed_blind
    both["true_pos"] = ~both.passed_truth if POSITIVE_IS_FAIL else both.passed_truth
    positive_label = "fail (non-conformant)" if POSITIVE_IS_FAIL else "pass (conformant)"

    # Metrics: all months together, then month by month
    overall = pd.DataFrame([classification_metrics(both.pred_pos, both.true_pos)], index=["all months"])
    monthly = pd.DataFrame({f"{y}-{m:02d}": classification_metrics(g.pred_pos, g.true_pos)
                            for (y, m), g in both.groupby(["year", "month"])}).T
    print(f"Positive class = {positive_label}; reference = ground truth; tolerance ±{SLOPE_TOLERANCE_PCT}%")
    display(pd.concat([overall, monthly]).round(3))

    # Export the misclassified site-months (FP = blind flags positive, truth doesn't; FN = the reverse)
    both["outcome"] = np.select(
        [both.pred_pos & ~both.true_pos, ~both.pred_pos & both.true_pos],
        ["false_positive", "false_negative"], default="correct")
    misclassified = both.loc[both.outcome != "correct"].sort_values(["outcome", "site_id", "year", "month"])
    csv_path = OUT_DIR / f"pass_fail_misclassified_tol{SLOPE_TOLERANCE_PCT}.csv"
    misclassified.drop(columns=["pred_pos", "true_pos"]).to_csv(csv_path, index=False)
    print(f"{(misclassified.outcome == 'false_positive').sum():,} FP and "
          f"{(misclassified.outcome == 'false_negative').sum():,} FN site-months "
          f"({misclassified.site_id.nunique():,} distinct sites) -> {csv_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Conformance rate -- BLIND results only. Denominator = assessable site-months (fit_status == "ok").
# ---------------------------------------------------------------------------
assert SLOPE_TOLERANCE_PCT is not None, "Set SLOPE_TOLERANCE_PCT in Section 1 and run the pass/fail cell first."

MIN_GROUP_SITES = 10        # hide groups smaller than this in the displayed tables (noisy rates)
SITE_PASS_SHARE = 1       # site-level: conformant if it passes in >= this share of its assessable months
INSTALL_GROUPING = "year"   # "year" or "month" (month-year of pv_install_date)
STANDARD_MANDATORY = pd.Timestamp("2021-12-18")  # AS/NZS 4777.2:2020 mandatory for new installs

# Blind pass/fail + site attributes
meta = site_meta[["site_id", "manufacturer", "dnsp_name", "pv_install_date"]].rename(
    columns={"manufacturer": "oem", "dnsp_name": "dnsp", "pv_install_date": "install_date"})
pf = blind_pf[["site_id", "year", "month", "passed"]].merge(meta, on="site_id", how="left")
pf[["oem", "dnsp"]] = pf[["oem", "dnsp"]].fillna("Unknown")
pf["install"] = (pf.install_date.dt.year if INSTALL_GROUPING == "year"
                 else pf.install_date.dt.to_period("M").astype(str))
pf["install_era"] = np.where(pf.install_date >= STANDARD_MANDATORY,
                             "installed after 2020 standard mandatory", "installed before")

# Funnel per month: eligible -> with data -> assessable -> conformant
funnel = blind.groupby(["year", "month"]).agg(
    eligible_sites=("site_id", "nunique"),
    with_data=("fit_status", lambda s: (s != "no_data").sum()),
    assessable=("fit_status", lambda s: (s == "ok").sum()),
)
funnel["conformant"] = pf.groupby(["year", "month"]).passed.sum()
funnel["conformant_%_of_assessable"] = (100 * funnel.conformant / funnel.assessable).round(1)
display(funnel)

In [ ]:
def rate_table(df: pd.DataFrame, by, flag: str = "passed", min_sites: int = MIN_GROUP_SITES) -> pd.DataFrame:
    """Conformance rate per group: n sites, n rows (site-months or sites), % conformant. Small groups hidden."""
    g = df.groupby(by)
    out = pd.DataFrame({"n_sites": g.site_id.nunique(), "n_rows": g.size(),
                        "conformant_%": (100 * g[flag].mean()).round(1)})
    hidden = (out.n_sites < min_sites).sum()
    if hidden:
        print(f"({hidden} group(s) with < {min_sites} sites hidden)")
    return out.loc[out.n_sites >= min_sites].sort_values("n_sites", ascending=False)

# Whole of years, site-month basis
all_rate = 100 * pf.passed.mean()
print(f"All years overall: {all_rate:.1f}% of {len(pf):,} assessable site-months "
      f"({pf.site_id.nunique():,} sites) conformant at ±{SLOPE_TOLERANCE_PCT}% slope tolerance")

# Site-level: one verdict per site across its assessable months
sites_pf = (pf.groupby("site_id")
              .agg(n_months=("passed", "size"), share_pass=("passed", "mean"),
                   oem=("oem", "first"), dnsp=("dnsp", "first"),
                   install=("install", "first"), install_era=("install_era", "first"))
              .reset_index())
sites_pf["conformant"] = sites_pf.share_pass >= SITE_PASS_SHARE
print(f"Site level: {100 * sites_pf.conformant.mean():.1f}% of {len(sites_pf):,} sites conformant "
      f"(pass in >= {SITE_PASS_SHARE:.0%} of their assessable months)")
display(sites_pf.n_months.describe().round(1).to_frame("assessable months per site"))

In [ ]:
# Each breakdown at two levels: site-months (all years) and sites (one verdict per site)
BREAKDOWNS = [("OEM", "oem"), ("DNSP", "dnsp"),
              (f"Install {INSTALL_GROUPING}", "install"), ("Install era", "install_era")]
SORT_BY_GROUP = {"install", "install_era"}   # chronological order; the others by size (largest first)

for label, col in BREAKDOWNS:
    print(f"\n=== Conformance by {label} ===")
    table = rate_table(pf, col).join(
        rate_table(sites_pf, col, flag="conformant", min_sites=0)[["conformant_%"]]
            .rename(columns={"conformant_%": "site_level_conformant_%"}),
        how="left")
    if col in SORT_BY_GROUP:
        table = table.sort_index()             # e.g. 2009 -> 2025
    display(table.rename(columns={"n_rows": "n_site_months", "conformant_%": "site_month_conformant_%"}))

In [ ]:
# ---------------------------------------------------------------------------
# site-level conformance by install year
# (site conformant only if it passes in all its assessable months)
# ---------------------------------------------------------------------------
yr = rate_table(sites_pf, "install", flag="conformant").sort_index()   # years with >= MIN_GROUP_SITES sites

x = np.arange(len(yr))
fig, ax = plt.subplots(figsize=(12, 3.4), dpi=300, layout="constrained")
bars = ax.bar(x, yr["conformant_%"], 0.65, color="#1f4e79")
ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=1)

# Standard became mandatory on 18 Dec 2021: line between the 2021 and 2022 install years
first_after = STANDARD_MANDATORY.year + (1 if STANDARD_MANDATORY.month == 12 else 0)
if first_after in yr.index:
    xm = list(yr.index).index(first_after) - 0.5
    ax.axvline(xm, color="#374151", ls="--", lw=1)
    ax.text(xm + 0.1, yr["conformant_%"].max() * 1.12, "AS/NZS 4777.2:2020\nmandatory →",
            fontsize=7, va="top", color="#374151")

ax.set_xticks(x, [f"{int(y)}\n(n={n:,})" for y, n in zip(yr.index, yr.n_sites)], fontsize=7)
ax.set_xlabel("Installation year (n = sites)", fontsize=8)
ax.set_ylabel("Sites conformant in all\nassessable months (%)", fontsize=8)
ax.set_ylim(0, yr["conformant_%"].max() * 1.15)
ax.tick_params(axis="y", labelsize=7)
ax.grid(axis="y", color="#ebebeb", lw=0.5)
ax.set_axisbelow(True)

fig.savefig(OUT_DIR / "site_conformance_by_install_year.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "site_conformance_by_install_year.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Site-level conformance by OEM x install year (one verdict per site, see SITE_PASS_SHARE)
# ---------------------------------------------------------------------------
MIN_CELL_SITES = 10   # blank out OEM-year cells with fewer sites than this (rate too noisy)

oem_year = sites_pf.merge(meta[["site_id", "install_date"]], on="site_id", how="left")
oem_year["install_year"] = oem_year.install_date.dt.year

big_oems = rate_table(sites_pf, "oem", flag="conformant").index   # OEMs with >= MIN_GROUP_SITES sites
oem_year = oem_year.loc[oem_year.oem.isin(big_oems)]

# Two tables with the same shape: % conformant and the number of sites behind each cell
pct = oem_year.pivot_table(index="oem", columns="install_year", values="conformant", aggfunc="mean") * 100
n = oem_year.pivot_table(index="oem", columns="install_year", values="site_id", aggfunc="nunique")
pct = pct.where(n >= MIN_CELL_SITES).round(1)

order = n.sum(axis=1).sort_values(ascending=False).index   # largest OEMs first
print(f"Site-level conformant % by OEM x install year (cells with < {MIN_CELL_SITES} sites blank)")
display(pct.loc[order])
print("Number of sites per cell")
display(n.loc[order].fillna(0).astype(int))

# One OEM on its own, e.g. M13
OEM = "M13"
display(pd.DataFrame({"n_sites": n.loc[OEM], "site_level_conformant_%": pct.loc[OEM]}).dropna(subset=["n_sites"]))

In [ ]:
# ---------------------------------------------------------------------------
# Heatmap: site-level conformant % by OEM (rows) x install year (columns)
# ---------------------------------------------------------------------------

HEAT_VMAX = 100          # fixed 0-100% scale, so heatmaps from different runs compare directly
SHOW_N = True            # print the number of sites under each %

data = pct.loc[order]
counts = n.loc[order, data.columns]
years = list(data.columns)

cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#f0f0f0")  # blank (too few sites) cells: light grey, hatched below

fig, ax = plt.subplots(figsize=(0.75 * len(years) + 3, 0.45 * len(data) + 2), dpi=130)
im = ax.imshow(np.ma.masked_invalid(data.to_numpy(dtype=float)), cmap=cmap,
               norm=mcolors.Normalize(0, HEAT_VMAX), aspect="auto")

# Cell labels: dark text on light cells, white text on dark cells; hatch the blanks
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        value, n_sites = data.iat[i, j], counts.iat[i, j]
        if np.isnan(value):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="///",
                                       edgecolor="#c8c8c8", linewidth=0))
            continue
        text = f"{value:.0f}" + (f"\n({int(n_sites)})" if SHOW_N else "")
        ax.text(j, i, text, ha="center", va="center", fontsize=6.5,
                color="white" if value > 0.6 * HEAT_VMAX else "#1f2937")

# Where the 2020 standard became mandatory (between install years)
first_year_after = STANDARD_MANDATORY.year + (1 if STANDARD_MANDATORY.month == 12 else 0)
if first_year_after in years:
    x = years.index(first_year_after) - 0.5
    ax.axvline(x, color="#374151", linestyle="--", linewidth=1.2)
    ax.text(x + 0.1, -0.7, "AS/NZS 4777.2:2020 mandatory →", fontsize=7.5, color="#374151", va="bottom")

# Axes, white gridlines between cells, colorbar, legend for blanks
ax.set_xticks(range(len(years)), [str(y) for y in years], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(data)), data.index, fontsize=8)
ax.set_xticks(np.arange(-0.5, len(years)), minor=True)
ax.set_yticks(np.arange(-0.5, len(data)), minor=True)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="minor", length=0)
ax.set_xlabel("Install year")
ax.set_ylabel("Inverter manufacturer (OEM)")
ax.set_title(f"Site-level Volt-VAr conformance by OEM and install year\n"
             f"(% of sites conformant in all assessable months, ±{SLOPE_TOLERANCE_PCT}% slope tolerance; "
             f"(n) = sites)", fontsize=10, fontweight="bold", pad=18)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Site-level conformant (%)", fontsize=8)
ax.legend(handles=[Patch(facecolor="#f0f0f0", edgecolor="#c8c8c8", hatch="///",
                         label=f"< {MIN_CELL_SITES} sites")],
          loc="upper left", bbox_to_anchor=(1.12, 1.0), fontsize=7.5, frameon=False)

fig.tight_layout()
fig.savefig(OUT_DIR / f"heatmap_oem_install_year_tol{SLOPE_TOLERANCE_PCT}.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Heatmap: site-level conformant % by GROUP (rows) x install year (columns)
# ---------------------------------------------------------------------------

GROUP_COL = "dnsp"                      # "dnsp" or "oem"
GROUP_LABEL = {"dnsp": "DNSP", "oem": "Inverter manufacturer (OEM)"}[GROUP_COL]
HEAT_VMAX = 100          # fixed 0-100% scale, so heatmaps from different runs compare directly
SHOW_N = True            # print the number of sites under each %

# Site-level verdicts with install year, restricted to groups with >= MIN_GROUP_SITES sites
g = sites_pf.merge(meta[["site_id", "install_date"]], on="site_id", how="left")
g["install_year"] = g.install_date.dt.year
g = g.loc[g[GROUP_COL].isin(rate_table(sites_pf, GROUP_COL, flag="conformant").index)]

pct = g.pivot_table(index=GROUP_COL, columns="install_year", values="conformant", aggfunc="mean") * 100
n = g.pivot_table(index=GROUP_COL, columns="install_year", values="site_id", aggfunc="nunique")
order = n.sum(axis=1).sort_values(ascending=False).index          # largest groups first
data = pct.where(n >= MIN_CELL_SITES).loc[order]                   # blank cells with too few sites
counts = n.loc[order, data.columns]
years = list(data.columns)

cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#f0f0f0")  # blank (too few sites) cells: light grey, hatched below

fig, ax = plt.subplots(figsize=(0.75 * len(years) + 3, 0.45 * len(data) + 2), dpi=130)
im = ax.imshow(np.ma.masked_invalid(data.to_numpy(dtype=float)), cmap=cmap,
               norm=mcolors.Normalize(0, HEAT_VMAX), aspect="auto")

# Cell labels: dark text on light cells, white text on dark cells; hatch the blanks
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        value, n_sites = data.iat[i, j], counts.iat[i, j]
        if np.isnan(value):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, hatch="///",
                                       edgecolor="#c8c8c8", linewidth=0))
            continue
        text = f"{value:.0f}" + (f"\n({int(n_sites)})" if SHOW_N else "")
        ax.text(j, i, text, ha="center", va="center", fontsize=6.5,
                color="white" if value > 0.6 * HEAT_VMAX else "#1f2937")

# Where the 2020 standard became mandatory (between install years)
first_year_after = STANDARD_MANDATORY.year + (1 if STANDARD_MANDATORY.month == 12 else 0)
if first_year_after in years:
    x = years.index(first_year_after) - 0.5
    ax.axvline(x, color="#374151", linestyle="--", linewidth=1.2)
    ax.text(x + 0.1, -0.7, "AS/NZS 4777.2:2020 mandatory →", fontsize=7.5, color="#374151", va="bottom")

# Axes, white gridlines between cells, colorbar, legend for blanks
ax.set_xticks(range(len(years)), [str(int(y)) for y in years], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(data)), data.index, fontsize=8)
ax.set_xticks(np.arange(-0.5, len(years)), minor=True)
ax.set_yticks(np.arange(-0.5, len(data)), minor=True)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="minor", length=0)
ax.set_xlabel("Install year")
ax.set_ylabel(GROUP_LABEL)
ax.set_title(f"Site-level Volt-VAr conformance by {GROUP_LABEL.split(' (')[0]} and install year\n"
             f"(% of sites conformant in all assessable months, ±{SLOPE_TOLERANCE_PCT}% slope tolerance; "
             f"(n) = sites)", fontsize=10, fontweight="bold", pad=18)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Site-level conformant (%)", fontsize=8)
ax.legend(handles=[Patch(facecolor="#f0f0f0", edgecolor="#c8c8c8", hatch="///",
                         label=f"< {MIN_CELL_SITES} sites")],
          loc="upper left", bbox_to_anchor=(1.12, 1.0), fontsize=7.5, frameon=False)

fig.tight_layout()
fig.savefig(OUT_DIR / f"heatmap_{GROUP_COL}_install_year_tol{SLOPE_TOLERANCE_PCT}.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Heatmaps: site-level conformant % by install year -- OEM (top) and DNSP (bottom)
# ---------------------------------------------------------------------------

HEAT_VMAX = 100
SHOW_N = True

# Site-level verdicts with install year (one row per site)
sites_year = sites_pf.merge(
    meta[["site_id", "install_date"]],
    on="site_id",
    how="left"
)
sites_year["install_year"] = sites_year.install_date.dt.year


def group_year_tables(col: str):
    """% conformant and n sites per (group, install year)."""
    keep = rate_table(
        sites_year,
        col,
        flag="conformant"
    ).index

    d = sites_year.loc[sites_year[col].isin(keep)]

    pct = (
        d.pivot_table(
            index=col,
            columns="install_year",
            values="conformant",
            aggfunc="mean"
        ) * 100
    )

    n = d.pivot_table(
        index=col,
        columns="install_year",
        values="site_id",
        aggfunc="nunique"
    )

    order = n.sum(axis=1).sort_values(ascending=False).index

    return pct.where(n >= MIN_CELL_SITES).loc[order], n.loc[order]


panels = {
    "Inverter manufacturer (OEM)": group_year_tables("oem"),
    "DNSP": group_year_tables("dnsp"),
}

# Same install-year columns in both panels
years = sorted(
    set().union(*(p.columns for p, _ in panels.values()))
)

panels = {
    k: (
        p.reindex(columns=years),
        c.reindex(columns=years)
    )
    for k, (p, c) in panels.items()
}

cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#f0f0f0")

norm = mcolors.Normalize(0, HEAT_VMAX)

first_year_after = (
    STANDARD_MANDATORY.year
    + (1 if STANDARD_MANDATORY.month == 12 else 0)
)

row_counts = [len(p) for p, _ in panels.values()]

fig, axes = plt.subplots(
    len(panels),
    1,
    sharex=True,
    dpi=130,
    figsize=(
        0.5 * len(years) + 3,
        0.45 * sum(row_counts) + 3
    ),
    gridspec_kw={
        "height_ratios": row_counts,
        "hspace": 0.08,  # reduce this further, e.g. 0.02, if desired
    }
)

for ax, (ylabel, (data, counts)) in zip(axes, panels.items()):
    im = ax.imshow(
        np.ma.masked_invalid(data.to_numpy(dtype=float)),
        cmap=cmap,
        norm=norm,
        aspect="auto"
    )

    # Cell labels and hatched blank cells
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data.iat[i, j]
            n_sites = counts.iat[i, j]

            if np.isnan(value):
                ax.add_patch(
                    plt.Rectangle(
                        (j - 0.5, i - 0.5),
                        1,
                        1,
                        fill=False,
                        hatch="///",
                        edgecolor="#c8c8c8",
                        linewidth=0
                    )
                )
                continue

            text = f"{value:.0f}"

            if SHOW_N:
                text += f"\n({int(n_sites)})"

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=6.5,
                color=(
                    "white"
                    if value > 0.6 * HEAT_VMAX
                    else "#1f2937"
                )
            )

    # Mandatory-standard marker
    if first_year_after in years:
        ax.axvline(
            years.index(first_year_after) - 0.5,
            color="#374151",
            linestyle="--",
            linewidth=1.2
        )

    # Gridlines between cells
    ax.set_yticks(
        range(len(data)),
        data.index,
        fontsize=8
    )

    ax.set_xticks(
        np.arange(-0.5, len(years)),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, len(data)),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=1.5
    )

    ax.tick_params(
        which="minor",
        length=0
    )

    ax.set_ylabel(ylabel)

# Shared x-axis
axes[-1].set_xticks(
    range(len(years)),
    [str(y) for y in years],
    rotation=45,
    ha="right",
    fontsize=8
)

axes[-1].set_xlabel("Install year")

# Mandatory-date label
if first_year_after in years:
    axes[0].text(
        years.index(first_year_after) - 0.4,
        -0.7,
        "AS/NZS 4777.2:2020 mandatory →",
        fontsize=7.5,
        color="#374151",
        va="bottom"
    )

axes[0].set_title(
    "Site-level Volt-VAr conformance by install year\n"
    f"(% of sites conformant in all assessable months, "
    f"±{SLOPE_TOLERANCE_PCT}% slope tolerance; (n) = sites)",
    fontsize=10,
    fontweight="bold",
    pad=20
)

# Shared colorbar and legend
cbar = fig.colorbar(
    im,
    ax=axes,
    fraction=0.025,
    pad=0.02
)

cbar.set_label(
    "Site-level conformant (%)",
    fontsize=8
)

axes[0].legend(
    handles=[
        Patch(
            facecolor="#f0f0f0",
            edgecolor="#c8c8c8",
            hatch="///",
            label=f"< {MIN_CELL_SITES} sites"
        )
    ],
    loc="lower left",
    bbox_to_anchor=(1.0, 1.02),
    fontsize=7.5,
    frameon=False
)

fig.savefig(
    OUT_DIR
    / f"heatmap_oem_dnsp_install_year_tol"
      f"{SLOPE_TOLERANCE_PCT}.png",
    bbox_inches="tight"
)

plt.show()

## 7. Sensitivity (one month)

### 7.1.1. GATE test

In [ ]:
SENS_MONTH = "2025-01"
SENS_Y, SENS_M = int(SENS_MONTH[:4]), int(SENS_MONTH[-2:])
SENSITIVITY = {
    "gate_15pct":   dict(FIT_SETTINGS, p_min_frac=0.15),
    "gate_30pct":   dict(FIT_SETTINGS, p_min_frac=0.30),
    "night_00_03":  dict(FIT_SETTINGS, night_hours=(0, 3)),
}

rows = []
truth_month = truth.loc[(truth.year == SENS_Y) & (truth.month == SENS_M)]
for name, settings in {"main": FIT_SETTINGS, **SENSITIVITY}.items():
    out = OUT_DIR / "blind" if name == "main" else OUT_DIR / "sensitivity" / name
    Fleet.run_months([SENS_MONTH], out, sites, memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **settings)
    res = Fleet.add_fit_status(Fleet.load_results(out), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
    res = res.loc[(res.year == SENS_Y) & (res.month == SENS_M)]
    cmp = Fleet.compare_to_truth(res, truth_month)
    rows.append({"setting": name, "n_ok": (res.fit_status == "ok").sum(),
                 "median_slope": res.loc[res.fit_status == "ok", "slope"].median(),
                 "median_abs_err_vs_truth": cmp.slope_err.abs().median()})
display(pd.DataFrame(rows).round(3))

### 7.2. SLOPE test

In [ ]:
# ---------------------------------------------------------------------------
# Sensitivity: slope tolerance around the required Volt-VAr slope.
# Reclassifies stored fits (main 242-253 V window, all months); no refitting.
# ---------------------------------------------------------------------------
REQ = Fleet.REQUIRED_SLOPE                 # -60 % / 18 V = -3.333 %/V
TOLS = [0.10, 0.15, 0.20, 0.25, 0.30]
MAIN_TOL = 0.20


def ok_fits(df):
    """Retained fits only: site_id, year, month, slope."""
    if "fit_status" not in df.columns:
        df = Fleet.add_fit_status(df, min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
    return df.loc[df.fit_status == "ok", ["site_id", "year", "month", "slope"]]


blind_all = ok_fits(Fleet.load_results(OUT_DIR / "blind"))
truth_all = ok_fits(truth)

both = blind_all.merge(truth_all, on=["site_id", "year", "month"], suffixes=("_blind", "_truth"))
print(f"Unmonitored assessable: {len(blind_all):,} site-months at {blind_all.site_id.nunique():,} sites")
print(f"Reference assessable:   {len(truth_all):,} site-months at {truth_all.site_id.nunique():,} sites")
print(f"Overlap:                {len(both):,} site-months at {both.site_id.nunique():,} sites")


def band(tol):
    return REQ * (1 + tol), REQ * (1 - tol)     # (lower, upper), e.g. (-4.000, -2.667)


def site_rate(df, passed):
    """Share of sites conformant in every assessable month."""
    return passed.groupby(df.site_id.values).all().mean()


rows = []
for tol in TOLS:
    lo, hi = band(tol)
    u = both.slope_blind.between(lo, hi)
    r = both.slope_truth.between(lo, hi)
    tp, fp = (u & r).sum(), (u & ~r).sum()
    fn, tn = (~u & r).sum(), (~u & ~r).sum()

    fb = blind_all.slope.between(lo, hi)        # fleet, unmonitored
    ft = truth_all.slope.between(lo, hi)        # fleet, reference

    rows.append({
        "tolerance": f"±{tol:.0%}",
        "band (%/V)": f"[{lo:.3f}, {hi:.3f}]",
        # agreement on the U–R overlap
        "agreement": (tp + tn) / len(both),
        "precision": tp / (tp + fp),
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "F1": 2 * tp / (2 * tp + fp + fn),
        "U pass (overlap)": u.mean(),
        "R pass (overlap)": r.mean(),
        # fleet conformance
        "U site-month": fb.mean(),
        "U site": site_rate(blind_all, fb),
        "R site-month": ft.mean(),
        "R site": site_rate(truth_all, ft),
        # disagreements near a band edge (within 0.3 %/V)
        "borderline share of disagreements": (
            ((both.slope_blind - lo).abs().lt(0.3) | (both.slope_blind - hi).abs().lt(0.3) |
             (both.slope_truth - lo).abs().lt(0.3) | (both.slope_truth - hi).abs().lt(0.3))[u != r].mean()),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })

sens_tol = pd.DataFrame(rows)
display(sens_tol.round(3))

# Check: the ±20% row should reproduce the main results (94.5% agreement, 21.2% / 13.6% fleet)
main = sens_tol.loc[sens_tol.tolerance == f"±{MAIN_TOL:.0%}"].iloc[0]
print(f"\n±20% check: agreement {main.agreement:.1%}, "
      f"U site-month {main['U site-month']:.1%}, U site {main['U site']:.1%}")

sens_tol.to_csv(OUT_DIR / "sensitivity" / "slope_tolerance.csv", index=False)

## 8. Save outputs

In [ ]:
# One tidy table for later notebooks / the paper: every site-month, both sources, with status
results = pd.concat([blind, truth], ignore_index=True)
results.to_parquet(OUT_DIR / "site_month_results.parquet", index=False)
eligibility.to_parquet(OUT_DIR / "site_eligibility.parquet", index=False)
print(f"Saved {len(results):,} rows -> {OUT_DIR / 'site_month_results.parquet'}")
print(f"Saved {len(eligibility):,} rows -> {OUT_DIR / 'site_eligibility.parquet'}")

In [ ]:
print(f"{100 * (sites_pf.share_pass > 0).mean():.1f}%")